<a href="https://colab.research.google.com/github/alessiobsc/AI4Cyber-FaceRecognitionSecurity/blob/main/AI4Cyber_ProjectWork_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount= True)

In [ ]:
!pip install adversarial-robustness-toolbox # ART libreria per generare e valutare attacchi adversarial
!pip install torch torchvision
!pip install facenet_pytorch

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

# Import da ART
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.defences.preprocessor import SpatialSmoothing

# Verifica che la GPU sia disponibile
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo in uso: {device}")

# Fase 1

In [ ]:
import pandas as pd
import subprocess
import os

# ==========================================
# 1. CONFIGURAZIONE PERCORSI
# ==========================================
CSV_PATH = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/identity_meta.csv'
ARCHIVIO_DRIVE = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/vggface2_train.tar.gz'
ARCHIVIO_LOCALE = '/content/vgg_temp.tar.gz'
OUTPUT_LOCAL_DIR = '/content/dataset_progetto'
DESTINAZIONE_DRIVE = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/Dataset_Selezionato_Progetto'

ID_CERCI = '250' # Verrà gestito come stringa per matchare il dataset

# ==========================================
# 2. SELEZIONE LOGICA (FLAG = 1)
# ==========================================
print("--- FASE 1: Selezione Identità (Metodo Clean) ---")
df = pd.read_csv(CSV_PATH, on_bad_lines='skip')

# Pulizia: rimuoviamo spazi e assicuriamoci che Class_ID sia stringa
df.columns = df.columns.str.strip()
df['Class_ID'] = df['Class_ID'].astype(str).str.strip()
df['Gender'] = df['Gender'].str.strip()

# Se Cerci non viene trovato, cerchiamo con il prefisso 'n' tipico del dataset.
if ID_CERCI not in df['Class_ID'].values:
    ID_CERCI = 'n' + ID_CERCI.zfill(6)

# FILTRO LOGICO: Solo soggetti presenti nel Training Set (Flag == 1)
df_train = df[df['Flag'] == 1].copy()

# Stratified Sampling (50 Maschi, 50 Femmine)
df_m = df_train[df_train['Gender'] == 'm']
df_f = df_train[df_train['Gender'] == 'f']

# Assicuriamoci che Cerci sia nel pool maschile
if ID_CERCI not in df_m['Class_ID'].values:
    print(f"ATTENZIONE: L'ID {ID_CERCI} non è stato trovato nel Train Set con Gender='m'.")
    males_sampled = df_m.sample(n=50, random_state=42)['Class_ID'].tolist()
else:
    males_sampled = df_m[df_m['Class_ID'] != ID_CERCI].sample(n=49, random_state=42)['Class_ID'].tolist()
    males_sampled.append(ID_CERCI)

females_sampled = df_f.sample(n=50, random_state=42)['Class_ID'].tolist()

identita_selezionate = males_sampled + females_sampled
cartelle_da_estrarre = [f"train/{id_val}" for id_val in identita_selezionate]

print(f"Selezionate {len(identita_selezionate)} identità dal Training Set.")

# ==========================================
# 3. ESTRAZIONE VELOCE (LOCAL SSD)
# ==========================================
print("\n--- FASE 2: Estrazione Veloce ---")
os.makedirs(OUTPUT_LOCAL_DIR, exist_ok=True)

print("1. Copia dell'archivio da Drive a Colab...")
!cp "{ARCHIVIO_DRIVE}" "{ARCHIVIO_LOCALE}"

print("2. Estrazione selettiva dal disco locale...")
comando_tar = ['tar', '-xzf', ARCHIVIO_LOCALE, '-C', OUTPUT_LOCAL_DIR] + cartelle_da_estrarre
subprocess.run(comando_tar)

print("3. Pulizia file temporaneo...")
if os.path.exists(ARCHIVIO_LOCALE): os.remove(ARCHIVIO_LOCALE)

# ==========================================
# 4. SALVATAGGIO PERMANENTE
# ==========================================
print("\n--- FASE 3: Sincronizzazione con Drive ---")
!cp -r {OUTPUT_LOCAL_DIR} "{DESTINAZIONE_DRIVE}"

# Verifica finale
percorso_output = os.path.join(OUTPUT_LOCAL_DIR, 'train')
if os.path.exists(percorso_output):
    final_count = len(os.listdir(percorso_output))
    print(f"Missione compiuta! Identità totali nel dataset finale: {final_count}")
else:
    print("Errore: la cartella di output è vuota.")

In [ ]:
#Riduce il dataset selezionato mantenendo esattamente 10 immagini per identità
import os
import glob
import random
import shutil

# ==========================================
# 1. POTATURA LOCALE
# ==========================================
DATASET_DIR = '/content/dataset_progetto/train'
DESTINAZIONE_DRIVE = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/Dataset_Selezionato_Progetto'
IMM_PER_IDENTITA = 10

random.seed(42) # Per riproducibilità

print(f"1. Controllo cartella locale: {DATASET_DIR}")
if os.path.exists(DATASET_DIR):
    cartelle = [f.path for f in os.scandir(DATASET_DIR) if f.is_dir()]
    print(f"Trovate {len(cartelle)} identità. Inizio potatura a {IMM_PER_IDENTITA} immagini ciascuna...")

    immagini_rimosse = 0
    for cartella in cartelle:
        immagini = glob.glob(os.path.join(cartella, '*.*'))
        if len(immagini) > IMM_PER_IDENTITA:
            da_mantenere = set(random.sample(immagini, IMM_PER_IDENTITA))
            for img_path in immagini:
                if img_path not in da_mantenere:
                    os.remove(img_path)
                    immagini_rimosse += 1

    print(f"Potatura completata! Rimosse {immagini_rimosse} immagini in eccesso.")

    # ==========================================
    # 2. SALVATAGGIO SU DRIVE
    # ==========================================
    print("\n2. Inizio sincronizzazione del mini-dataset su Drive...")

    if os.path.exists(DESTINAZIONE_DRIVE):
        print("Rilevata cartella precedente su Drive, pulizia in corso per sicurezza...")
        shutil.rmtree(DESTINAZIONE_DRIVE)

    # Copia veloce dei 1000 file rimanenti
    os.system(f'cp -r /content/dataset_progetto "{DESTINAZIONE_DRIVE}"')

    # Verifica finale su Drive
    if os.path.exists(os.path.join(DESTINAZIONE_DRIVE, 'train')):
        conteggio = sum([len(files) for r, d, files in os.walk(os.path.join(DESTINAZIONE_DRIVE, 'train'))])
        print(f"\Il dataset definitivo ha esattamente {conteggio} immagini salvate su Drive.")
    else:
        print("\nErrore durante la copia finale su Drive.")
else:
    print("\nERRORE CRITICO: La cartella locale non esiste.")

In [ ]:
#100 idendità e che ogni cartella contenga esattamente 10 immagini, per un totale di 100 immagini
import os

path_drive = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/Dataset_Selezionato_Progetto/train'

if os.path.exists(path_drive):
    cartelle = [d for d in os.listdir(path_drive) if os.path.isdir(os.path.join(path_drive, d))]

    report_errori = []
    totale_immagini = 0

    for c in cartelle:
        p_completo = os.path.join(path_drive, c)
        n_foto = len([f for f in os.listdir(p_completo) if os.path.isfile(os.path.join(p_completo, f))])
        totale_immagini += n_foto

        if n_foto != 10:
            report_errori.append(f"ID {c}: {n_foto} immagini")

    print(f"--- VERIFICA INTEGRITÀ ---")
    print(f"Identità totali: {len(cartelle)}")
    print(f"Immagini totali: {totale_immagini}")

    if len(report_errori) == 0 and totale_immagini == 1000:
        print("\nPERFETTO: Tutte le 100 identità hanno esattamente 10 immagini.")
    else:
        print("\nANOMALIA RILEVATA nelle seguenti cartelle:")
        for err in report_errori:
            print(err)
else:
    print("Errore: Il percorso su Drive non è raggiungibile.")

In [ ]:
import pandas as pd

# 1. Recuperiamo automaticamente gli ID dalle cartelle che hai appena verificato
path_drive = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/Dataset_Selezionato_Progetto/train'

if os.path.exists(path_drive):
    # La lista delle cartelle corrisponde alla lista degli ID (es. 'n000001')
    selected_ids = [d for d in os.listdir(path_drive) if os.path.isdir(os.path.join(path_drive, d))]
    print(f"Estratti con successo {len(selected_ids)} ID dalle directory.")
else:
    print("Errore: impossibile accedere alle cartelle su Drive.")
    selected_ids = []

# 2. Imposta il percorso del file CSV contenente i metadati di VGG-Face2
csv_meta_path = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/identity_meta.csv'

try:
    # Carichiamo il dataset
    df_meta = pd.read_csv(csv_meta_path, on_bad_lines='skip')

    # Pulizia preventiva dei nomi delle colonne
    df_meta.columns = df_meta.columns.str.strip()

    # Adattiamo i nomi delle colonne allo standard VGG-Face2
    col_id = 'Class_ID'
    col_name = 'Name'

    # Puliamo gli ID nel dataframe rimuovendo eventuali spazi e apici
    df_meta[col_id] = df_meta[col_id].astype(str).str.strip().str.replace('"', '').str.replace("'", "")

    # 3. Filtriamo il dataframe mantenendo solo le righe il cui ID è presente nel nostro dataset di 100 soggetti
    df_filtrato = df_meta[df_meta[col_id].isin(selected_ids)]

    print(f"\n--- MAPPA IDENTITÀ ({len(df_filtrato)} trovate) ---\n")
    print(df_filtrato[[col_id, col_name]].to_string(index=False))

except FileNotFoundError:
    print(f"\n[ERRORE] Il file '{csv_meta_path}' non è stato trovato.")
except KeyError as e:
    print(f"\n[ERRORE] Colonna non trovata: {e}")
    print(f"Colonne disponibili nel CSV: {list(df_meta.columns)}")

# Fase 2


In [ ]:
from facenet_pytorch import InceptionResnetV1
#Carica un modello già allenato sul dataset VGGFace2 e lo mette in fase di inferenza
resnet = InceptionResnetV1(pretrained='vggface2').eval()
#modalità classificazione diretta
resnet.classify = True

In [ ]:
resnet.to(device)

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
import tensorflow as tf

# ==========================================
# 1. SETUP MODELLO E LABELS
# ==========================================
print("Inizializzazione modello InceptionResnetV1 (NN1)...")
resnet = InceptionResnetV1(pretrained='vggface2').eval()
resnet.classify = True

# Caricamento delle etichette (VGGFace2) del classificatore
fpath = tf.keras.utils.get_file('rcmalli_vggface_labels_v2.npy',
                             "https://github.com/rcmalli/keras-vggface/releases/download/v2.0/rcmalli_vggface_labels_v2.npy",
                             cache_subdir="./")
LABELS = np.load(fpath)

# ==========================================
# 2. INTEGRAZIONE METADATI (identity_meta.csv)
# ==========================================
# Sostituisci con il path reale del tuo file csv
csv_path = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/identity_meta.csv'

try:
    # 1. skipinitialspace=True: rimuove gli spazi bianchi insidiosi dopo le virgole
    # 2. on_bad_lines='skip': se una riga ha più colonne del previsto (es. riga 652), la salta
    #    e continua a leggere il resto del file senza andare in crash.
    meta_df = pd.read_csv(csv_path, skipinitialspace=True, on_bad_lines='skip')

    # Pulizia colonne e dati
    meta_df.columns = meta_df.columns.str.strip()
    meta_df['Class_ID'] = meta_df['Class_ID'].astype(str).str.strip().str.replace('"', '').str.replace("'", "")

    # Dizionario: ID -> Nome
    id_to_name = dict(zip(meta_df['Class_ID'], meta_df['Name']))
    print(f"Metadati caricati con successo.")

except Exception as e:
    print(f"Errore caricamento CSV metadati: {e}")
    id_to_name = {}

# ==========================================
# 3. PREPROCESSING MATEMATICO
# ==========================================
preprocess = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    # Normalizzazione lineare da [0, 1] a [-1, 1] per allinearsi all'addestramento originale
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# ==========================================
# 4. LOOP DI VALUTAZIONE (CORRETTO)
# ==========================================
# Sostituisci con il path della tua cartella train se diverso
path_drive = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/Dataset_Selezionato_Progetto/train'

correct_predictions = 0
total_images = 0
error_log = []

print("\nInizio classificazione baseline sul Test Set...")

with torch.no_grad():
    for folder_name in os.listdir(path_drive):
        folder_path = os.path.join(path_drive, folder_name)
        if not os.path.isdir(folder_path):
            continue

        # Otteniamo il VERO NOME dalla cartella usando il nostro dizionario
        # Rimuoviamo eventuali spazi e virgolette per sicurezza
        real_name = id_to_name.get(folder_name, "ID_Sconosciuto").strip().replace('"', '').replace("'", "")

        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)

            try:
                # Caricamento e preprocessing immagine
                img = Image.open(img_path).convert('RGB')
                img_tensor = preprocess(img).unsqueeze(0)

                # Forward pass
                logits = resnet(img_tensor)

                # Predizione
                pred_idx = logits.argmax(dim=1).item()
                # Il file rcmalli_vggface_labels_v2.npy restituisce direttamente i nomi (es. ' Fabio_Fognini')
                pred_name = str(LABELS[pred_idx]).strip().replace('"', '').replace("'", "")

                # Check Ground Truth vs Prediction confrontando i NOMI!
                if real_name == pred_name:
                    correct_predictions += 1
                else:
                    if len(error_log) < 15:
                        error_log.append(f"Vero: {real_name} (ID: {folder_name}) --> Predetto: {pred_name}")

                total_images += 1
                if total_images % 100 == 0:
                    print(f"Analizzate {total_images} immagini...")

            except Exception as e:
                print(f"Errore su {img_path}: {e}")

# ==========================================
# 5. METRICHE
# ==========================================
accuracy = correct_predictions / total_images if total_images > 0 else 0

print(f"\n======================================")
print(f"RISULTATI BASELINE EVALUATION")
print(f"======================================")
print(f"Totale Immagini : {total_images}")
print(f"Corrette (TP)   : {correct_predictions}")
print(f"Accuratezza     : {accuracy * 100:.2f}%")
print(f"======================================\n")

if error_log:
    print("Campione degli errori di classificazione:")
    for err in error_log:
        print(" -", err)

# Fase 3

# Load FGSM samples

In [ ]:
# Utility import
utils_path = "/content/drive/MyDrive/Gruppo IA4Cyber"
if utils_path not in sys.path: sys.path.append(utils_path)

from cyber_utils import load_attack_results


## FGSM Untargeted

In [ ]:
# 1. Carica i dati (assicurati che il percorso sia corretto)
folder_fgsm_untargeted = '/content/drive/MyDrive/Gruppo IA4Cyber/FGSM/untargeted_BEST'
print("=== Analisi FGSM Untargeted ===")
risultati_fgsm = load_attack_results(folder_fgsm_untargeted)

# 2. Selezioniamo il file con epsilon 0.1 per l'analisi
if 'eps_0_1' in risultati_fgsm:
    run_data = risultati_fgsm['eps_0_1']
    y_true = run_data['y_true']
    preds_adv = run_data['preds_adv']

    # Calcoliamo la classe predetta (quella con la probabilità/logit massimo)
    y_pred = np.argmax(preds_adv, axis=1)

    # 3. Metriche Globali
    acc = accuracy_score(y_true, y_pred)
    asr = 1.0 - acc # L'Attack Success Rate in untargeted è 1 - Accuracy

    print("\n--- Metriche Globali (eps=0.1) ---")
    print(f"Accuracy residua : {acc * 100:.2f}%")
    print(f"Attack Success Rate: {asr * 100:.2f}%")

    # 4. Matrice di Confusione (limitata alle prime N classi per leggibilità)
    # Mostriamo solo un sottoinsieme, ad es. i primi 10 soggetti,
    # altrimenti 100x100 diventa illeggibile nel notebook.

    N_CLASSI_PLOT = 10
    classi_uniche = np.unique(y_true)[:N_CLASSI_PLOT]

    # Filtriamo y_true e y_pred per mostrare solo le immagini di queste 10 classi
    mask = np.isin(y_true, classi_uniche)
    y_true_sub = y_true[mask]
    y_pred_sub = y_pred[mask]

    cm = confusion_matrix(y_true_sub, y_pred_sub, labels=classi_uniche)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
                xticklabels=classi_uniche, yticklabels=classi_uniche)
    plt.title(f"Matrice di Confusione - FGSM (eps=0.1)\nPrimi {N_CLASSI_PLOT} Soggetti", fontsize=14)
    plt.xlabel("Predizione del Modello (Sotto Attacco)")
    plt.ylabel("Classe Reale (Ground Truth)")
    plt.show()
else:
    print("File 'eps_0_1' non trovato nel dizionario.")

## FGSM Targeted

In [ ]:
import os
import numpy as np

# 1. Percorso della cartella Targeted (Cerci)
folder_targeted = '/content/drive/MyDrive/Gruppo IA4Cyber/FGSM/targeted__Alessio_Cerci'

print("=== Caricamento Dati FGSM Targeted ===")
risultati_tg = load_attack_results(folder_targeted)

# 2. Selezioniamo il file
nome_file_tg = 'eps_0_1'

if nome_file_tg in risultati_tg:
    data_tg = risultati_tg[nome_file_tg]

    x_adv_tg = data_tg['x_adv']
    y_true_tg = data_tg['y_true']
    preds_adv_tg = data_tg['preds_adv']
    target_idx = data_tg['target_idx']
    target_name = data_tg['target_name']

    # Calcolo delle predizioni (indice della classe con probabilità massima)
    y_pred_tg = np.argmax(preds_adv_tg, axis=1)

    # Calcolo ASR: quante immagini sono diventate 'Cerci'?
    successi = np.sum(y_pred_tg == target_idx)
    asr = (successi / len(y_true_tg)) * 100

    print(f"\n--- Analisi Targeted verso: {target_name} ---")
    print(f"Target Index: {target_idx}")
    print(f"Attack Success Rate (ASR): {asr:.2f}% ({successi}/1000 immagini)")
else:
    print(f"File {nome_file_tg} non trovato. Chiavi disponibili: {list(risultati_tg.keys())}")

In [ ]:
import tensorflow as tf
import pandas as pd

# Caricamento delle etichette (VGGFace2) del classificatore
fpath = tf.keras.utils.get_file('rcmalli_vggface_labels_v2.npy',
                             "https://github.com/rcmalli/keras-vggface/releases/download/v2.0/rcmalli_vggface_labels_v2.npy",
                             cache_subdir="./")
LABELS = np.load(fpath)

path_drive = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/Dataset_Selezionato_Progetto/train'

csv_meta_path = '/content/drive/MyDrive/Colab Notebooks/AI4Cyber/PW/identity_meta.csv'

try:
    meta_df = pd.read_csv(csv_meta_path, skipinitialspace=True, on_bad_lines='skip')
    meta_df.columns = meta_df.columns.str.strip()
    meta_df['Class_ID'] = meta_df['Class_ID'].astype(str).str.strip().str.replace('"', '').str.replace("'", "")

    # Crea il dizionario che mappa, ad esempio, 'n000156' -> ' Ajda_Pekkan'
    id_to_name = dict(zip(meta_df['Class_ID'], meta_df['Name']))
    print(f"Metadati caricati con successo.")
except Exception as e:
    print(f"Errore caricamento CSV metadati: {e}")
    id_to_name = {}

In [ ]:
# Prepara il test set in formato compatibile con ART.
# Ogni immagine viene ridimensionata, convertita in tensore e normalizzata
# nel range [-1, 1]. Le etichette testuali vengono convertite negli indici
# numerici usati dal classificatore VGGFace2.

import numpy as np

# Mapping inverso: Nome Persona -> Indice numerico del modello
name_to_idx = {name.strip().replace("'", ""): i for i, name in enumerate(LABELS)}

x_test_list = []
y_test_list = []

print("Caricamento delle 1000 immagini del test set...")

for folder_name in os.listdir(path_drive):
    folder_path = os.path.join(path_drive, folder_name)
    if not os.path.isdir(folder_path): continue

    real_name = id_to_name.get(folder_name, "").strip().replace("'", "")
    label_idx = name_to_idx.get(real_name, -1)
    if label_idx == -1: continue

    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        try:
            # Preprocessing identico alla baseline (78.5%)
            img_pil = Image.open(img_path).convert('RGB').resize((160, 160))
            img_tns = transforms.ToTensor()(img_pil)
            # Normalizzazione [-1, 1] come deciso
            img_tns = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])(img_tns)

            x_test_list.append(img_tns.numpy())
            y_test_list.append(label_idx)
        except:
            continue

x_test_all = np.array(x_test_list)
y_test_all = np.array(y_test_list)

print(f"Completato! Dataset pronto: {x_test_all.shape}")

In [ ]:
def show_targeted_forensics(x_orig, x_adv, y_true, y_pred, target_idx, target_name, num_images=3):
    # 1. Trova tutti gli indici dove l'attacco ha avuto successo
    indici_successo = np.where((y_pred == target_idx) & (y_true != target_idx))[0]

    if len(indici_successo) == 0:
        print("Nessuna immagine è stata convertita con successo al target.")
        return

    # 2. Logica per selezionare soggetti diversi
    indici_da_mostrare = []
    soggetti_visti = set()

    for idx in indici_successo:
        classe_reale = y_true[idx]
        if classe_reale not in soggetti_visti:
            indici_da_mostrare.append(idx)
            soggetti_visti.add(classe_reale)

        # Ci fermiamo quando abbiamo raggiunto il numero desiderato
        if len(indici_da_mostrare) == num_images:
            break

    # 3. Visualizzazione
    plt.figure(figsize=(15, 5 * len(indici_da_mostrare)))

    for i, idx in enumerate(indici_da_mostrare):
        img_orig = (x_orig[idx].transpose(1, 2, 0) + 1) / 2
        img_adv = (x_adv[idx].transpose(1, 2, 0) + 1) / 2

        noise = np.abs(img_adv - img_orig)
        noise_visual = np.clip(noise * 15, 0, 1)

        plt.subplot(len(indici_da_mostrare), 3, i * 3 + 1)
        plt.imshow(np.clip(img_orig, 0, 1))
        plt.title(f"Soggetto: {LABELS[y_true[idx]]}\n(Classe Reale: {y_true[idx]})")
        plt.axis('off')

        plt.subplot(len(indici_da_mostrare), 3, i * 3 + 2)
        plt.imshow(np.clip(img_adv, 0, 1))
        plt.title(f"Targeted -> {target_name}", color='green', fontweight='bold')
        plt.axis('off')

        plt.subplot(len(indici_da_mostrare), 3, i * 3 + 3)
        plt.imshow(noise_visual)
        plt.title("Pattern di Rumore (x15)")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Esegui con la nuova funzione
show_targeted_forensics(x_test_all, x_adv_tg, y_true_tg, y_pred_tg, target_idx, target_name)

In [ ]:
import seaborn as sns

plt.figure(figsize=(12, 6))
# Contiamo quante volte ogni classe è stata predetta
unique, counts = np.unique(y_pred_tg, return_counts=True)
prediction_counts = dict(zip(unique, counts))

# Creiamo un barplot
sns.barplot(x=list(prediction_counts.keys())[:20], y=list(prediction_counts.values())[:20])
plt.axhline(y=len(y_test_all)/len(LABELS), color='r', linestyle='--', label='Distribuzione Uniforme')
plt.title(f"Distribuzione delle Predizioni dopo Attacco Targeted verso {target_name}")
plt.xlabel("ID Classe")
plt.ylabel("Numero di Predizioni")
plt.show()